# Gold — `dbr_caresync.gold.lab_test_trends`

One row per test type (test_name) per calendar month. Aggregates lab_results by test name, year and month. Joins to calendar for month/quarter labels. Includes LAG-based month-over-month abnormal rate change. Full rebuild on every run.

In [0]:
# ── Ensure gold schema exists ─────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS dbr_caresync.gold")

# ── Build gold.lab_test_trends ───────────────────────────────────────────────────
spark.sql("""
CREATE OR REPLACE TABLE dbr_caresync.gold.lab_test_trends AS
WITH base AS (
    SELECT
        test_name,
        YEAR(test_date)  AS year,
        MONTH(test_date) AS month,
        COUNT(*)                                                          AS total_tests,
        COUNT(CASE WHEN result_status = 'Abnormal' THEN 1 END)           AS abnormal_test_count,
        COUNT(CASE WHEN result_status = 'Critical' THEN 1 END)           AS critical_test_count
    FROM dbr_caresync.silver.lab_results
    GROUP BY test_name, YEAR(test_date), MONTH(test_date)
),
with_rates AS (
    SELECT
        b.test_name,
        b.year,
        b.month,
        c.month_name,
        c.quarter_name,
        b.total_tests,
        b.abnormal_test_count,
        b.critical_test_count,
        CAST(b.abnormal_test_count * 100.0 / b.total_tests AS DECIMAL(5, 2))  AS abnormal_rate_pct,
        CAST(b.critical_test_count * 100.0 / b.total_tests AS DECIMAL(5, 2))  AS critical_rate_pct
    FROM base b
    LEFT JOIN dbr_caresync.silver.calendar c
        ON CAST(DATE_FORMAT(MAKE_DATE(b.year, b.month, 1), 'yyyyMMdd') AS INT) = c.date_key
),
with_lag AS (
    SELECT
        *,
        LAG(abnormal_rate_pct) OVER (
            PARTITION BY test_name ORDER BY year, month
        ) AS prev_month_abnormal_rate_pct
    FROM with_rates
)
SELECT
    test_name,
    year,
    month,
    month_name,
    quarter_name,
    total_tests,
    abnormal_test_count,
    critical_test_count,
    abnormal_rate_pct,
    critical_rate_pct,
    prev_month_abnormal_rate_pct,
    CASE
        WHEN prev_month_abnormal_rate_pct IS NULL THEN NULL
        ELSE CAST(abnormal_rate_pct - prev_month_abnormal_rate_pct AS DECIMAL(6, 2))
    END AS abnormal_rate_change_pct
FROM with_lag
""")

print("\u2705 Table created: dbr_caresync.gold.lab_test_trends")
result = spark.table("dbr_caresync.gold.lab_test_trends")
print(f"   Rows    : {result.count():,}")
print(f"   Columns : {len(result.columns)}")
display(result.limit(5))

In [0]:
record_count = result.count()
dbutils.notebook.exit(str(record_count))